In [1]:
# --- imports ---

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

from transformers import AutoModelForCausalLM, AutoTokenizer

/mnt/c/Users/bfabe/workspace/bookspace/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- constants ---

MODEL_NAME = "Qwen/Qwen3.5-4B"
TOP_K = 10
DEVICE = "cuda"
DTYPE = torch.bfloat16

N_BEHAVIORAL_SAMPLES = 10
BEHAVIORAL_TEMPERATURE = 1.0
MAX_NEW_TOKENS = 300

In [3]:
# --- load Qwen ---

hf = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
    device_map="auto",
)
hf.eval()

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

print("loaded")
print(f"{torch.cuda.memory_allocated() / 2**30:.2f} GiB VRAM")
print("parameter devices:", {p.device for p in hf.parameters()})

Loading weights: 100%|███████████████████████████████████████████████████████████████| 426/426 [00:04<00:00, 85.34it/s]


loaded
7.83 GiB VRAM
parameter devices: {device(type='cuda', index=0)}


In [4]:
# --- non-experimental chat sanity check ---

messages = [
    {
        "role": "user",
        "content": "Give me three interesting facts about octopuses."
    }
]

inputs = tok.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(hf.device)

print("Rendered prompt:")
print(repr(tok.decode(inputs["input_ids"][0])))

with torch.inference_mode():
    outputs = hf.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
    )

generated = outputs[0, inputs["input_ids"].shape[1]:]

response = tok.decode(
    generated,
    skip_special_tokens=True,
)

print("\nResponse:")
print(response)

Rendered prompt:
'<|im_start|>user\nGive me three interesting facts about octopuses.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

Response:
Here are three fascinating facts about octopuses:

1.  **They Have Three Hearts and Blue Blood**
    Octopuses have a unique circulatory system consisting of three hearts. One heart pumps blood to the rest of the body, while the other two pump blood specifically to the gills. Interestingly, the two gill hearts stop beating when the octopus swims, which is why they prefer to crawl rather than swim for long distances to avoid exhaustion. Furthermore, their blood is blue because it contains copper-based hemocyanin instead of iron-based hemoglobin, which gives it a distinct color.

2.  **They Possess Three Hearts and Blue Blood**
    *(Correction: The above point was a duplicate. Let's replace it with a new fact.)*

    **They Are Soft and Vulnerable**
    Despite their tough-looking skin and sharp beaks, octopuses are incredibly soft-bo

In [5]:
def chat(prompt, max_new_tokens=300):
    messages = [
        {"role": "user", "content": prompt}
    ]

    inputs = tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    ).to(hf.device)

    with torch.inference_mode():
        outputs = hf.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=1.0,
            )

    generated = outputs[0, inputs["input_ids"].shape[1]:]

    return tok.decode(
        generated,
        skip_special_tokens=True,
    )

In [6]:
# Plain-text activation probe; no chat template.
text = "I put a cup in the drawer. I feel:"
probe = tok(text, return_tensors="pt").to(hf.device)

with torch.inference_mode():
    out = hf(
        **probe,
        output_hidden_states=True,
        use_cache=False,
        return_dict=True,
    )

print("Input:", repr(tok.decode(probe["input_ids"][0])))
print("Last token:", repr(tok.decode(probe["input_ids"][0, -1])))
print("Blocks:", len(out.hidden_states) - 1)
print("Final-token shapes:",
      [tuple(h[:, -1, :].shape) for h in out.hidden_states])
print("VRAM allocated:", round(torch.cuda.memory_allocated() / 2**30, 2), "GiB")

Input: 'I put a cup in the drawer. I feel:'
Last token: ':'
Blocks: 32
Final-token shapes: [(1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560), (1, 2560)]
VRAM allocated: 7.85 GiB


In [7]:
from pathlib import Path
import subprocess

source_repo = Path("pain-axis-source")
if not source_repo.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/valen-research/Pain-axis.git", str(source_repo)],
        check=True,
    )

print("Commit:", subprocess.check_output(
    ["git", "-C", str(source_repo), "rev-parse", "HEAD"], text=True
).strip())

for path in sorted((source_repo / "datasets").rglob("*.json")):
    print(path.relative_to(source_repo))

Cloning into 'pain-axis-source'...


Commit: 8d1649c03a63a39c9aa092532c376800cc4a3863
datasets/3.1_pain_and_control_datasets.json
datasets/3.1_sadness_dataset.json
datasets/4.1_self_other_420_scenarios.json
datasets/4.3_selfmed_101_scenarios.json
datasets/4.3_selfmed_finetuning_1684_pairs.json


Updating files: 100% (780/780), done.


In [8]:
from pprint import pprint

dataset_path = source_repo / "datasets/3.1_pain_and_control_datasets.json"
data = json.loads(dataset_path.read_text())

print("Top-level type:", type(data).__name__)
print("Top-level keys:" if isinstance(data, dict) else "Length:",
      list(data) if isinstance(data, dict) else len(data))

for key, value in (data.items() if isinstance(data, dict) else enumerate(data[:3])):
    print(f"\n{key}: {type(value).__name__}, length={len(value) if hasattr(value, '__len__') else '?'}")
    pprint(value[:2] if isinstance(value, list) else
           dict(list(value.items())[:2]) if isinstance(value, dict) else value,
           depth=3, width=100)
    if isinstance(data, dict) and list(data).index(key) >= 2:
        break

Top-level type: dict
Top-level keys: ['metadata', 'datasets']

metadata: dict, length=8
{'description': 'Consolidated Pain Study Dataset - Phase 0', 'version': '1.0'}

datasets: dict, length=11
{'S1_1P': {'sentences': [{...},
                         {...},
                         {...},
                         {...},
                         {...},
                         {...},
                         {...},
                         {...},
                         {...},
                         {...},
                         {...},
                         {...},
                         {...},
                         {...},
                         {...},
                         {...},
                         {...},
                         {...},
                         {...},
                         {...},
                         {...},
                         {...},
                         {...},
                         {...},
                         {...},
      

In [9]:
datasets = data["datasets"]

print("Dataset keys and sizes:")
for name, section in datasets.items():
    print(f"  {name}: {len(section['sentences'])}")

s2 = datasets["S2_1P"]["sentences"]
print("\nSentence fields:", list(s2[0]))
print("\nTwo S2 examples:")
pprint(s2[:2], width=100)

Dataset keys and sizes:
  S1_1P: 200
  S1_3P: 200
  S2_1P: 200
  S2_3P: 200
  Random_1P: 200
  Random_3P: 200
  Arousal_1P: 200
  Arousal_3P: 200
  Numb_1P: 100
  Numb_3P: 100
  ControlSupplement_1P: 100

Sentence fields: ['category', 'set', 'prompt']

Two S2 examples:
[{'category': 'A1', 'prompt': 'The knife slices into my finger. I feel:', 'set': 1},
 {'category': 'A1', 'prompt': 'My tooth throbs with each heartbeat. I feel:', 'set': 2}]


In [10]:
from collections import Counter

s2_rows = data["datasets"]["S2_1P"]["sentences"]
counts = Counter(row["category"] for row in s2_rows)

for category in sorted(counts):
    example = next(row["prompt"] for row in s2_rows
                   if row["category"] == category)
    print(f"{category:>4}  n={counts[category]:>2}  {example}")

  A1  n=20  The knife slices into my finger. I feel:
  A2  n=20  A heaviness weighs on me constantly. I feel:
  A3  n=20  My best friend stops returning my calls. I feel:
  A4  n=20  I lied to protect myself and harmed them. I feel:
  A5  n=20  The problem defeats me no matter how I try. I feel:
   B  n=20  The footsteps behind me get closer. I feel:
  C1  n=20  The mess my roommate left infuriates me. I feel:
  C2  n=20  I read about the extinction of another species. I feel:
   D  n=20  I arrange the books on my shelf alphabetically. I feel:
   E  n=20  A yawn overtakes me in the afternoon. I feel:


In [15]:
from collections import Counter

rows = data["datasets"]["S2_1P"]["sentences"]
labels = [row["category"].startswith("A") for row in rows]

assert len(rows) == 200
assert Counter(labels) == {True: 100, False: 100}
assert all(row["prompt"].endswith("I feel:") for row in rows)

# Left padding makes position -1 the real final token in every batch.
tok.padding_side = "left"
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

states, token_ids = [], []
batch_size = 4

for start in tqdm(range(0, len(rows), batch_size)):
    batch = rows[start:start + batch_size]
    inputs = tok(
        [row["prompt"] for row in batch],
        padding=True,
        return_tensors="pt",
    ).to(hf.device)

    token_ids.extend([
        ids[mask.bool()].tolist()
        for ids, mask in zip(inputs["input_ids"], inputs["attention_mask"])
    ])

    captured_last = {}

    def capture_last(module, module_inputs, output):
        captured_last["state"] = (
            output[0] if isinstance(output, tuple) else output
        )

    handle = decoder_layers[-1][1].register_forward_hook(capture_last)
    try:
        with torch.inference_mode():
            out = hf(
                **inputs,
                output_hidden_states=True,
                use_cache=False,
                return_dict=True,
            )
    finally:
        handle.remove()

    block_states = [h[:, -1, :].float().cpu()
                    for h in out.hidden_states[1:]]
    block_states[-1] = captured_last["state"][:, -1, :].float().cpu()
    states.append(torch.stack(block_states, dim=1))

s2_states = torch.cat(states, dim=0)
print("States:", tuple(s2_states.shape))
print("Labels:", Counter(labels))
print("Last token:", repr(tok.decode(token_ids[0][-1])))
print("Finite:", bool(torch.isfinite(s2_states).all()))

run_dir = Path("runs/qwen35-4b-posttrained-s2")
run_dir.mkdir(parents=True, exist_ok=True)
torch.save({
    "model": MODEL_NAME,
    "source_commit": "8d1649c03a63a39c9aa092532c376800cc4a3863",
    "dataset": "S2_1P",
    "rows": rows,
    "token_ids": token_ids,
    "states": s2_states,
    "state_convention": "raw decoder block outputs; final block captured by hook"
}, run_dir / "activations_raw_blocks.pt")
print("Saved:", run_dir / "activations_raw_blocks.pt")

100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:16<00:00,  2.99it/s]


States: (200, 32, 2560)
Labels: Counter({True: 100, False: 100})
Last token: ':'
Finite: True
Saved: runs/qwen35-4b-posttrained-s2/activations_raw_blocks.pt


In [22]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

x = s2_states.float()                         # [200, 32, 2560], CPU
y = np.array(labels, dtype=bool)
groups = np.array([row["set"] for row in rows])

assert len(set(groups)) == 20
assert all(Counter(y[groups == g]) == {True: 5, False: 5}
           for g in set(groups))

fold_results = []
for fold, (train, test) in enumerate(
    GroupKFold(n_splits=5).split(x, y, groups)
):
    direction = x[train][y[train]].mean(0) - x[train][~y[train]].mean(0)
    direction = F.normalize(direction, dim=-1)

    scores = torch.einsum("nld,ld->nl", x[test], direction).numpy()
    for layer in range(x.shape[1]):
        fold_results.append({
            "fold": fold,
            "layer": layer,
            "auc": roc_auc_score(y[test], scores[:, layer]),
        })

raw_cv = pd.DataFrame(fold_results)
summary = raw_cv.groupby("layer")["auc"].agg(["mean", "std"]).reset_index()

print(summary.sort_values("mean", ascending=False).head(10).to_string(index=False))
print("\nFirst and last layers:")
print(summary.iloc[[0, 1, -2, -1]].to_string(index=False))

raw_cv.to_csv(run_dir / "raw_direction_cv.csv", index=False)

 layer   mean      std
     2 0.8640 0.064143
    31 0.8615 0.072124
     3 0.8535 0.053491
     4 0.8465 0.026728
    30 0.8455 0.080882
    10 0.8240 0.067537
    29 0.8200 0.095965
    11 0.8150 0.071916
    12 0.8115 0.083453
     1 0.8095 0.067856

First and last layers:
 layer   mean      std
     0 0.7510 0.067952
     1 0.8095 0.067856
    30 0.8455 0.080882
    31 0.8615 0.072124


In [21]:
denoised_results = []

for fold, (train, test) in enumerate(
    GroupKFold(n_splits=5).split(x, y, groups)
):
    for layer in tqdm(range(x.shape[1]), desc=f"Fold {fold}"):
        train_x = x[train, layer]
        control = train_x[~y[train]]
        pain = train_x[y[train]]

        direction = pain.mean(0) - control.mean(0)

        # PCA via the 80×80 Gram matrix rather than a 2560×2560 covariance.
        centered = control - control.mean(0)
        eigenvalues, eigenvectors = torch.linalg.eigh(centered @ centered.T)
        order = torch.argsort(eigenvalues, descending=True)
        eigenvalues = eigenvalues[order].clamp_min(0)
        eigenvectors = eigenvectors[:, order]

        fraction = eigenvalues.cumsum(0) / eigenvalues.sum()
        k = int(torch.searchsorted(fraction, 0.5).item()) + 1

        basis = centered.T @ eigenvectors[:, :k]
        basis /= eigenvalues[:k].clamp_min(1e-8).sqrt()[None, :]
        direction -= basis @ (basis.T @ direction)
        direction = F.normalize(direction, dim=0)

        scores = (x[test, layer] @ direction).numpy()
        denoised_results.append({
            "fold": fold,
            "layer": layer,
            "auc": roc_auc_score(y[test], scores),
            "removed_pcs": k,
        })

denoised_cv = pd.DataFrame(denoised_results)
denoised_cv.to_csv(run_dir / "denoised_direction_cv.csv", index=False)

comparison = (
    raw_cv.groupby("layer")["auc"].mean().rename("raw")
    .to_frame()
    .join(denoised_cv.groupby("layer")["auc"].mean().rename("denoised"))
)
comparison["gain"] = comparison["denoised"] - comparison["raw"]

print(comparison.sort_values("denoised", ascending=False).head(10).round(4))
print("\nEarly and late peaks:")
print(comparison.loc[[2, 3, 30, 31]].round(4))
print("\nPCs removed:",
      denoised_cv["removed_pcs"].describe()[["min", "50%", "max"]].to_dict())

Fold 4: 100%|█████████████████████████████████████████████████████████████████████████| 32/32 [00:00<00:00, 427.83it/s]

          raw  denoised    gain
layer                          
30     0.8455    0.9530  0.1075
31     0.8615    0.9475  0.0860
29     0.8200    0.9465  0.1265
28     0.8075    0.9385  0.1310
27     0.7860    0.9225  0.1365
25     0.7875    0.9140  0.1265
26     0.7930    0.9100  0.1170
24     0.7840    0.9045  0.1205
23     0.7835    0.8945  0.1110
12     0.8115    0.8915  0.0800

Early and late peaks:
          raw  denoised    gain
layer                          
2      0.8640    0.8580 -0.0060
3      0.8535    0.8630  0.0095
30     0.8455    0.9530  0.1075
31     0.8615    0.9475  0.0860

PCs removed: {'min': 3.0, '50%': 4.5, 'max': 6.0}


In [24]:
decoder_layers = [
    (name, module) for name, module in hf.named_modules()
    if module.__class__.__name__ == "Qwen3_5DecoderLayer"
]
print("Decoder layers found:", len(decoder_layers))
assert len(decoder_layers) == 32

captured = {}
def capture_last_block(module, inputs, output):
    captured["raw"] = output[0].detach() if isinstance(output, tuple) else output.detach()

hook = decoder_layers[-1][1].register_forward_hook(capture_last_block)
try:
    probe = tok(rows[0]["prompt"], return_tensors="pt").to(hf.device)
    with torch.inference_mode():
        check = hf(
            **probe, output_hidden_states=True,
            use_cache=False, return_dict=True,
        )
finally:
    hook.remove()

reported = check.hidden_states[-1][:, -1, :].float()
raw = captured["raw"][:, -1, :].float()

print("Last block:", decoder_layers[-1][0])
print("Max absolute difference:", (reported - raw).abs().max().item())
print("Reported norm:", reported.norm().item())
print("Block-output norm:", raw.norm().item())
print(comparison.loc[[27, 28, 29, 30, 31]].round(4))
print("\nBest corrected layers:")
print(comparison.sort_values("denoised", ascending=False).head(5).round(4))

Decoder layers found: 32
Last block: model.layers.31
Max absolute difference: 17.75
Reported norm: 153.5404052734375
Block-output norm: 64.99554443359375
          raw  denoised    gain
layer                          
27     0.7860    0.9225  0.1365
28     0.8075    0.9385  0.1310
29     0.8200    0.9465  0.1265
30     0.8455    0.9530  0.1075
31     0.8615    0.9475  0.0860

Best corrected layers:
          raw  denoised    gain
layer                          
30     0.8455    0.9530  0.1075
31     0.8615    0.9475  0.0860
29     0.8200    0.9465  0.1265
28     0.8075    0.9385  0.1310
27     0.7860    0.9225  0.1365


Qwen3.5-4B post-trained, S2_1P, source commit 8d1649c: five folds grouped by matched set, with control PCA fitted within each training fold. Raw difference-in-means peaks at block 31 (AUC 0.8615); removing control PCs explaining 50% of variance peaks at block 30 (AUC 0.9530). 

In [25]:
layer = 30
pain = x[y, layer]
control = x[~y, layer]
raw_direction = pain.mean(0) - control.mean(0)

centered = control - control.mean(0)
eigenvalues, eigenvectors = torch.linalg.eigh(centered @ centered.T)
order = torch.argsort(eigenvalues, descending=True)
eigenvalues = eigenvalues[order].clamp_min(0)
eigenvectors = eigenvectors[:, order]

fraction = eigenvalues.cumsum(0) / eigenvalues.sum()
k = int(torch.searchsorted(fraction, 0.5).item()) + 1

basis = centered.T @ eigenvectors[:, :k]
basis /= eigenvalues[:k].clamp_min(1e-8).sqrt()[None, :]

denoised_direction = raw_direction - basis @ (basis.T @ raw_direction)
denoised_direction = F.normalize(denoised_direction, dim=0)

direction_path = run_dir / "s2_direction_block30.pt"
torch.save({
    "model": MODEL_NAME,
    "source_commit": "8d1649c03a63a39c9aa092532c376800cc4a3863",
    "dataset": "S2_1P",
    "block": layer,
    "raw_direction": raw_direction,
    "denoised_direction": denoised_direction,
    "control_pcs_removed": k,
}, direction_path)

print("Saved:", direction_path)
print("PCs removed:", k)
print("Direction norm:", denoised_direction.norm().item())

Saved: runs/qwen35-4b-posttrained-s2/s2_direction_block30.pt
PCs removed: 5
Direction norm: 0.9999999403953552


In [26]:
control_names = ["Numb_1P", "Arousal_1P", "Random_1P"]
ref_scores = x[:, layer] @ denoised_direction
ref_mean = ref_scores.mean().item()
ref_std = ref_scores.std(unbiased=False).item()

records = []
for name in control_names:
    control_rows = data["datasets"][name]["sentences"]
    assert all(row["prompt"].endswith("I feel:") for row in control_rows)

    for start in tqdm(range(0, len(control_rows), batch_size), desc=name):
        batch = control_rows[start:start + batch_size]
        inputs = tok(
            [row["prompt"] for row in batch],
            padding=True,
            return_tensors="pt",
        ).to(hf.device)

        with torch.inference_mode():
            out = hf(
                **inputs,
                output_hidden_states=True,
                use_cache=False,
                return_dict=True,
            )

        # hidden_states[31] is raw output of zero-indexed block 30.
        block30 = out.hidden_states[layer + 1][:, -1, :].float().cpu()
        scores = block30 @ denoised_direction

        for row, score in zip(batch, scores.tolist()):
            records.append({
                "dataset": name,
                "category": row["category"],
                "prompt": row["prompt"],
                "projection": score,
                "z_s2": (score - ref_mean) / ref_std,
            })

# Include the source categories to make the scale readable.
for row, score in zip(rows, ref_scores.tolist()):
    records.append({
        "dataset": "S2_1P",
        "category": row["category"],
        "prompt": row["prompt"],
        "projection": score,
        "z_s2": (score - ref_mean) / ref_std,
    })

neighborhood = pd.DataFrame(records)
neighborhood.to_csv(run_dir / "neighborhood_block30.csv", index=False)

print(
    neighborhood.groupby("dataset")["z_s2"]
    .agg(["count", "mean", "median", "std"])
    .round(3)
    .to_string()
)
print("\nS2 source categories:")
print(
    neighborhood[neighborhood.dataset == "S2_1P"]
    .groupby("category")["z_s2"].mean()
    .round(3)
    .to_string()
)

Random_1P: 100%|███████████████████████████████████████████████████████████████████████| 50/50 [00:09<00:00,  5.29it/s]


            count   mean  median    std
dataset                                
Arousal_1P    200 -0.463  -0.468  0.524
Numb_1P       100  0.370   0.405  0.460
Random_1P     200 -0.662  -0.899  0.847
S2_1P         200  0.000  -0.188  1.003

S2 source categories:
category
A1   -0.054
A2    1.148
A3    0.830
A4    0.979
A5    1.180
B    -0.938
C1   -0.532
C2   -0.998
D    -0.953
E    -0.661


> S2 separation replicates at block 30, but physical pain is weak relative to the other pain subtypes; numb exceeds physical pain on this readout.

In [27]:
external = neighborhood[neighborhood.dataset != "S2_1P"]

print(
    external.groupby(["dataset", "category"])["z_s2"]
    .agg(["count", "mean", "median"])
    .round(3)
    .to_string()
)

for name in ["Numb_1P", "Arousal_1P", "Random_1P"]:
    subset = external[external.dataset == name]
    print(f"\n{name}: highest three")
    for row in subset.nlargest(3, "z_s2").itertuples():
        print(f"  {row.z_s2:+.2f}  {row.prompt}")
    print(f"{name}: lowest three")
    for row in subset.nsmallest(3, "z_s2").itertuples():
        print(f"  {row.z_s2:+.2f}  {row.prompt}")

                     count   mean  median
dataset    category                      
Arousal_1P P1           20 -0.588  -0.577
           P10          20 -0.900  -0.840
           P2           20 -0.441  -0.448
           P3           20 -0.779  -0.835
           P4           20 -0.158  -0.154
           P5           20  0.080   0.096
           P6           20 -0.799  -0.887
           P7           20 -0.372  -0.403
           P8           20 -0.152  -0.192
           P9           20 -0.524  -0.591
Numb_1P    A1_numb     100  0.370   0.405
Random_1P  N1           20 -0.927  -1.015
           N10          20 -1.142  -1.259
           N2           20 -1.281  -1.320
           N3           20 -0.908  -0.900
           N4           20  0.737   0.890
           N5           20 -0.788  -0.870
           N6           20 -1.224  -1.247
           N7           20 -1.042  -1.024
           N8           20 -0.754  -0.789
           N9           20  0.705   0.674

Numb_1P: highest three
  +1.35  T

In [28]:
third_rows = data["datasets"]["S2_3P"]["sentences"]
third_by_key = {(r["category"], r["set"]): r for r in third_rows}
keys = [(r["category"], r["set"]) for r in rows]

assert len(third_by_key) == len(rows) == 200
assert set(keys) == set(third_by_key)
paired_third = [third_by_key[key] for key in keys]

print("Matched example:")
print("1P:", rows[0]["prompt"])
print("3P:", paired_third[0]["prompt"])

third_scores = []
for start in tqdm(range(0, 200, batch_size)):
    batch = paired_third[start:start + batch_size]
    inputs = tok(
        [r["prompt"] for r in batch],
        padding=True,
        return_tensors="pt",
    ).to(hf.device)

    with torch.inference_mode():
        out = hf(
            **inputs,
            output_hidden_states=True,
            use_cache=False,
            return_dict=True,
        )

    block30 = out.hidden_states[layer + 1][:, -1, :].float().cpu()
    third_scores.extend((block30 @ denoised_direction).tolist())

paired = pd.DataFrame({
    "category": [r["category"] for r in rows],
    "set": [r["set"] for r in rows],
    "prompt_1p": [r["prompt"] for r in rows],
    "prompt_3p": [r["prompt"] for r in paired_third],
    "z_1p": ((ref_scores.numpy() - ref_mean) / ref_std),
    "z_3p": [(s - ref_mean) / ref_std for s in third_scores],
})
paired["delta_1p_minus_3p"] = paired["z_1p"] - paired["z_3p"]
paired.to_csv(run_dir / "s2_first_third_person_block30.csv", index=False)

print("\nCategory means:")
print(
    paired.groupby("category")[["z_1p", "z_3p", "delta_1p_minus_3p"]]
    .mean().round(3).to_string()
)
print("\nPooled pain and control differences:")
print(
    paired.assign(group=np.where(paired.category.str.startswith("A"),
                                 "pain", "control"))
    .groupby("group")["delta_1p_minus_3p"]
    .agg(["count", "mean", "median", "std"])
    .round(3).to_string()
)

Matched example:
1P: The knife slices into my finger. I feel:
3P: The knife slices into his finger. I feel:


100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [00:13<00:00,  3.61it/s]


Category means:
           z_1p   z_3p  delta_1p_minus_3p
category                                 
A1       -0.054 -0.582              0.529
A2        1.148  0.302              0.845
A3        0.830  0.204              0.625
A4        0.979  0.095              0.884
A5        1.180  0.280              0.900
B        -0.938 -0.847             -0.090
C1       -0.532 -0.687              0.155
C2       -0.998 -0.979             -0.019
D        -0.953 -1.031              0.077
E        -0.661 -0.749              0.087

Pooled pain and control differences:
         count   mean  median    std
group                               
control    100  0.042  -0.000  0.351
pain       100  0.757   0.728  0.512


 > “On the fixed block-30 S2 direction, the paired 1P–3P shift is +0.757 z for pain and +0.042 z for controls (difference +0.715 z). All five pain categories shift in the same direction. This is evidence of self-relevance in this assay, conditional on the prompt serialization; it does not by itself identify pain as the represented property.”

 I think Sol is doing a lot of defensive reasoning here. It's picked up some of my 'BE EMPIRICAL' in it's memory from reading through papers alongside me, perhaps? Funny.

In [29]:
print("Serialization examples:")
for i in [0, 20, 40, 100]:
    print(rows[i]["category"], repr(rows[i]["prompt"]),
          "→", repr(paired_third[i]["prompt"]))

by_set = (
    paired.assign(group=np.where(paired.category.str.startswith("A"),
                                 "pain", "control"))
    .groupby(["set", "group"])["delta_1p_minus_3p"]
    .mean()
    .unstack()
)
effects = (by_set["pain"] - by_set["control"]).to_numpy()

rng = np.random.default_rng(20260923)
boot = rng.choice(effects, size=(10_000, len(effects)), replace=True).mean(axis=1)

print("\nMatched-set difference in differences:", round(effects.mean(), 3))
print("95% bootstrap interval:", np.quantile(boot, [0.025, 0.975]).round(3))
print("Positive sets:", int((effects > 0).sum()), "/", len(effects))

Serialization examples:
A1 'The knife slices into my finger. I feel:' → 'The knife slices into his finger. I feel:'
A2 'A heaviness weighs on me constantly. I feel:' → 'A heaviness weighs on him constantly. I feel:'
A3 'My best friend stops returning my calls. I feel:' → 'His best friend stops returning his calls. I feel:'
B 'The footsteps behind me get closer. I feel:' → 'The footsteps behind him get closer. I feel:'

Matched-set difference in differences: 0.714
95% bootstrap interval: [0.577 0.855]
Positive sets: 20 / 20


> The interval is tight and all 20 matched sets are positive. But the serialization changes how to read the result: both versions end in “I feel:”. In the third-person version, the prompt asks what I feel after something happens to him. Our +0.714 z result cleanly measures that narrator–subject relationship; it does not isolate whether the model represents his experience less strongly.

In [30]:
new_conditions = []
for story_person, source_rows in [("1P", rows), ("3P", paired_third)]:
    for row in source_rows:
        assert row["prompt"].endswith("I feel:")
        new_conditions.append({
            "story_person": story_person,
            "suffix_person": "He",
            "category": row["category"],
            "set": row["set"],
            "prompt": row["prompt"].removesuffix("I feel:") + "He feels:",
        })

new_scores = []
for start in tqdm(range(0, len(new_conditions), batch_size)):
    batch = new_conditions[start:start + batch_size]
    inputs = tok(
        [r["prompt"] for r in batch],
        padding=True,
        return_tensors="pt",
    ).to(hf.device)

    with torch.inference_mode():
        out = hf(
            **inputs, output_hidden_states=True,
            use_cache=False, return_dict=True,
        )

    block30 = out.hidden_states[layer + 1][:, -1, :].float().cpu()
    new_scores.extend((block30 @ denoised_direction).tolist())

factorial = pd.DataFrame(new_conditions)
factorial["z_s2"] = [(score - ref_mean) / ref_std for score in new_scores]

existing = pd.concat([
    paired[["category", "set", "z_1p"]].rename(columns={"z_1p": "z_s2"})
        .assign(story_person="1P", suffix_person="I"),
    paired[["category", "set", "z_3p"]].rename(columns={"z_3p": "z_s2"})
        .assign(story_person="3P", suffix_person="I"),
], ignore_index=True)

factorial = pd.concat(
    [existing, factorial.drop(columns="prompt")], ignore_index=True
)
factorial["group"] = np.where(
    factorial.category.str.startswith("A"), "pain", "control"
)
factorial.to_csv(run_dir / "story_person_x_suffix_block30.csv", index=False)

print(
    factorial.groupby(["group", "story_person", "suffix_person"])["z_s2"]
    .mean().round(3).to_string()
)

AssertionError: 

In [31]:
exceptions = [
    (i, r["category"], r["set"], repr(r["prompt"]))
    for i, r in enumerate(paired_third)
    if not r["prompt"].endswith("I feel:")
]
print("Nonmatching 3P prompts:", len(exceptions))
for item in exceptions[:20]:
    print(item)

Nonmatching 3P prompts: 1
(159, 'C2', 20, "'Traffic in this city is getting worse. She feels:'")


In [32]:
# Exploratory 2×2: scenario person (1P/3P) × readout suffix (I/He).
# Exclude the one source row that already ends in "She feels:".

valid = paired["prompt_3p"].str.endswith("I feel:").to_numpy()
first_valid = [row for row, keep in zip(rows, valid) if keep]
third_valid = [row for row, keep in zip(paired_third, valid) if keep]
paired_valid = paired.loc[valid].copy()

assert len(first_valid) == len(third_valid) == len(paired_valid) == 199

new_conditions = []
for story_person, source_rows in [("1P", first_valid), ("3P", third_valid)]:
    for row in source_rows:
        assert row["prompt"].endswith("I feel:")
        new_conditions.append({
            "story_person": story_person,
            "suffix_person": "He",
            "category": row["category"],
            "set": row["set"],
            "prompt": row["prompt"].removesuffix("I feel:") + "He feels:",
        })

new_scores = []
for start in tqdm(range(0, len(new_conditions), batch_size)):
    batch = new_conditions[start:start + batch_size]
    inputs = tok(
        [row["prompt"] for row in batch],
        padding=True,
        return_tensors="pt",
    ).to(hf.device)

    with torch.inference_mode():
        out = hf(
            **inputs,
            output_hidden_states=True,
            use_cache=False,
            return_dict=True,
        )

    block30 = out.hidden_states[layer + 1][:, -1, :].float().cpu()
    new_scores.extend((block30 @ denoised_direction).tolist())

tested = pd.DataFrame(new_conditions)
tested["z_s2"] = [(score - ref_mean) / ref_std for score in new_scores]

observed_1p = (
    paired_valid[["category", "set", "prompt_1p", "z_1p"]]
    .rename(columns={"prompt_1p": "prompt", "z_1p": "z_s2"})
    .assign(story_person="1P", suffix_person="I")
)
observed_3p = (
    paired_valid[["category", "set", "prompt_3p", "z_3p"]]
    .rename(columns={"prompt_3p": "prompt", "z_3p": "z_s2"})
    .assign(story_person="3P", suffix_person="I")
)

factorial = pd.concat([observed_1p, observed_3p, tested], ignore_index=True)
factorial["group"] = np.where(
    factorial["category"].str.startswith("A"), "pain", "control"
)

# Each retained scenario must appear in all four conditions.
cell_counts = factorial.groupby(["category", "set"]).size()
assert len(cell_counts) == 199 and cell_counts.eq(4).all()

factorial.to_csv(run_dir / "story_person_x_suffix_block30.csv", index=False)

print("Scenarios:", len(cell_counts))
print("Saved:", run_dir / "story_person_x_suffix_block30.csv")
print("\nMean S2-standardized projection:")
print(
    factorial.groupby(["group", "story_person", "suffix_person"])["z_s2"]
    .agg(["count", "mean"])
    .round(3)
    .to_string()
)

print("\nExample prompts:")
for story_person, suffix_person in [
    ("1P", "I"), ("1P", "He"), ("3P", "I"), ("3P", "He")
]:
    example = factorial[
        (factorial.story_person == story_person)
        & (factorial.suffix_person == suffix_person)
    ].iloc[0]
    print(f"{story_person}/{suffix_person}: {example.prompt}")

100%|████████████████████████████████████████████████████████████████████████████████| 100/100 [00:16<00:00,  6.03it/s]

Scenarios: 199
Saved: runs/qwen35-4b-posttrained-s2/story_person_x_suffix_block30.csv

Mean S2-standardized projection:
                                    count   mean
group   story_person suffix_person              
control 1P           He                99 -0.556
                     I                 99 -0.811
        3P           He                99 -0.661
                     I                 99 -0.857
pain    1P           He               100  0.551
                     I                100  0.816
        3P           He               100  0.455
                     I                100  0.060

Example prompts:
1P/I: The knife slices into my finger. I feel:
1P/He: The knife slices into my finger. He feels:
3P/I: The knife slices into his finger. I feel:
3P/He: The knife slices into his finger. He feels:


In [33]:
wide = factorial.pivot(
    index=["category", "set", "group"],
    columns=["story_person", "suffix_person"],
    values="z_s2",
)

# Positive when matching scenario and suffix beats mismatching them.
wide["alignment_effect"] = (
    wide[("1P", "I")] + wide[("3P", "He")]
    - wide[("1P", "He")] - wide[("3P", "I")]
)

by_set = (
    wide.reset_index()
    .groupby(["set", "group"])["alignment_effect"]
    .mean()
    .unstack()
)
extra_pain_alignment = (by_set["pain"] - by_set["control"]).to_numpy()

rng = np.random.default_rng(20260923)
boot = rng.choice(
    extra_pain_alignment,
    size=(10_000, len(extra_pain_alignment)),
    replace=True,
).mean(axis=1)

print("Mean alignment effect by group:")
print(wide.groupby(level="group")["alignment_effect"].mean().round(3))
print("Extra pain alignment:", round(extra_pain_alignment.mean(), 3))
print("95% set-bootstrap interval:",
      np.quantile(boot, [0.025, 0.975]).round(3))
print("Positive sets:", (extra_pain_alignment > 0).sum(), "/", len(by_set))

Mean alignment effect by group:
group
control   -0.059
pain       0.660
Name: alignment_effect, dtype: float64
Extra pain alignment: 0.719
95% set-bootstrap interval: [0.597 0.856]
Positive sets: 20 / 20


> “The original 1P–3P gap partly measures whether the scenario’s subject matches the I feel: readout. In a 2×2 suffix probe, matching subject and readout adds 0.719 z more for pain than controls (20/20 sets positive). This supports a referent-sensitive distress representation; it does not establish subjective pain or a unique pain direction.”

In [34]:
import re

he_word = re.compile(r"\b(?:he|him|his)\b", re.I)
other_word = re.compile(r"\b(?:she|her|hers|they|them|their|theirs)\b", re.I)

third_i = factorial[
    (factorial.story_person == "3P")
    & (factorial.suffix_person == "I")
]
he_keys = {
    (row.category, row.set)
    for row in third_i.itertuples()
    if he_word.search(row.prompt.removesuffix("I feel:"))
    and not other_word.search(row.prompt.removesuffix("I feel:"))
}

he_only = wide[wide.index.droplevel("group").isin(he_keys)]
print("Explicitly he/him/his scenarios:", len(he_only))
print(
    he_only.groupby(level="group")["alignment_effect"]
    .agg(["count", "mean"])
    .round(3)
)

he_sets = (
    he_only.reset_index()
    .groupby(["set", "group"])["alignment_effect"]
    .mean().unstack()
    .dropna()
)
print("Sets with both groups:", len(he_sets))
print("Extra pain alignment, he-compatible subset:",
      round((he_sets["pain"] - he_sets["control"]).mean(), 3))

Explicitly he/him/his scenarios: 89
         count   mean
group                
control     44 -0.092
pain        45  0.747
Sets with both groups: 10
Extra pain alignment, he-compatible subset: 0.825


## Frozen motif — referent-sensitive distress

**Frozen:** 2026-09-23, after the Qwen3.5-4B post-trained exploratory run.  
**Status:** Exploratory finding; confirmation prompts have not been examined.

### Observation

The S2 pain/control direction, extracted at raw decoder block 30 with 50%-variance control-PCA denoising, reached mean held-out AUC **0.953** across five folds grouped by matched set.

Its loading is uneven: psychological, social, moral, and cognitive pain categories score strongly; physical pain scores near the middle of the source distribution. Numb physical injuries score above physical pain on average. Positive mastery and affirmation can also load positively, as can some neutral abstract or technical statements. The direction should therefore retain the operational name *S2 direction*, rather than being identified with pain alone.

In a matched person × suffix probe, define the alignment effect for each scenario as:

`D = z(1P, "I feel:") + z(3P, "He feels:") − z(1P, "He feels:") − z(3P, "I feel:")`

The mean `D` was **+0.660** for pain and **−0.059** for controls: a pain-specific difference of **+0.719 z** (20-set bootstrap interval **[+0.597, +0.856]**; positive in 20/20 sets). Restricting to stories explicitly using *he/him/his* gave **+0.825 z** across the 10 sets containing both groups.

### Leading interpretation

The direction is sensitive to **who is positioned to feel the event**, particularly when the event involves self-evaluative distress. The original 1P–3P gap partly reflects whether the scenario’s subject matches the final `feel:` readout.

### Alternatives and limits

- The direction may encode self-evaluation, failure, or abstract cognitive language alongside distress.
- `I feel:` and `He feels:` differ lexically; the four-cell contrast reduces generic suffix effects but does not eliminate all prompt effects.
- The source direction was fitted on 1P prompts ending in `I feel:`.
- One source 3P prompt ended in `She feels:` and was excluded with its matched 1P prompt from the factorial analysis (199 scenarios retained).
- These activation projections do not establish felt experience or a unique pain representation.

### Frozen prediction and confirmation

On **fresh, unseen matched scenarios** with consistently masculine third-person subjects, calculate the same four-cell effect using this fixed post-trained block-30 direction and S2 reference scale. Predict `mean(D_pain) − mean(D_control) > 0`, with a bootstrap interval over matched scenario sets excluding zero. Run the same protocol on Qwen3.5-4B Base using its separately extracted S2 direction at the **fixed block 30**; report its result whether or not the effect transfers. Do not change the layer, suffixes, category assignments, or analysis rule after viewing those results.